In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
print("hii")

hii


In [2]:
!pip install -q datasets
!pip install -q transformers
!pip install -q evaluate
!pip install -q accelerate
!pip install -q tqdm
!pip install -q scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [2]:
pip show transformers

Name: transformers
Version: 5.0.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer-slim
Required-by: kaggle-environments, peft, sentence-transformers
Note: you may need to restart the kernel to use updated packages.


In [ ]:
print("HII")

In [50]:
!pip -q install evaluate
!pip -q install psutil

In [5]:
import gc
import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import psutil
import torch
import evaluate
import pynvml

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

warnings.filterwarnings("ignore")

In [ ]:
print("="*60)

print("Torch :", torch.__version__)

import transformers

print("Transformers :", transformers.__version__)

print("CUDA :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("="*60)

In [8]:
#######################################################

SEED = 42

set_seed(SEED)

#######################################################

MODELS = {

    "BERT-base": "bert-base-uncased",

    "DistilBERT": "distilbert-base-uncased"

}

#######################################################

TASKS = [

    "sst2",

    "qnli",

    "qqp",

    "wnli",

    "mnli",

    "mrpc",

    "rte"

]

#######################################################

MAX_LENGTH = 128

BATCH_SIZE = 16

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.10

NUM_EPOCHS = 5

#######################################################

In [9]:
TASK_TO_KEYS = {

    "sst2": ("sentence", None),

    "qnli": ("question", "sentence"),

    "qqp": ("question1", "question2"),

    "wnli": ("sentence1", "sentence2"),

    "mnli": ("premise", "hypothesis"),

    "mrpc": ("sentence1", "sentence2"),

    "rte": ("sentence1", "sentence2")

}

In [10]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(

        labels,

        predictions,

        average="weighted",

        zero_division=0

    )

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1

    }

In [11]:
def tokenize_function(examples, tokenizer, task):

    sentence1_key, sentence2_key = TASK_TO_KEYS[task]

    if sentence2_key is None:

        return tokenizer(

            examples[sentence1_key],

            truncation=True,

            max_length=MAX_LENGTH

        )

    return tokenizer(

        examples[sentence1_key],

        examples[sentence2_key],

        truncation=True,

        max_length=MAX_LENGTH

    )

In [12]:
results=[]

overall_start=time.time()

In [ ]:
experiment = 0

TOTAL_EXPERIMENTS = len(TASKS) * len(MODELS)

for task in TASKS:

    print("\n" + "="*90)
    print(f"TASK : {task.upper()}")
    print("="*90)

    dataset = load_dataset("glue", task)

    # MNLI has a different validation split
    if task == "mnli":
        validation_split = "validation_matched"
    else:
        validation_split = "validation"

    # Number of labels
    if task == "mnli":
        num_labels = 3
    else:
        num_labels = dataset["train"].features["label"].num_classes

    for model_name, checkpoint in MODELS.items():

        experiment += 1

        print("\n")
        print("-"*90)
        print(f"Experiment : {experiment}/{TOTAL_EXPERIMENTS}")
        print(f"Task       : {task.upper()}")
        print(f"Model      : {model_name}")
        print("-"*90)

        experiment_start = time.time()

In [141]:
        model = AutoModelForSequenceClassification.from_pretrained(

            checkpoint,

            num_labels=num_labels

        )

In [ ]:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)

        encoded_dataset = dataset.map(

            lambda x: tokenize_function(

                x,

                tokenizer,

                task

            ),

            batched=True

        )

        data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
        training_args = TrainingArguments(

            output_dir=f"./results/{task}_{model_name}",

            do_train=True,

            do_eval=True,

            eval_strategy="epoch",

            save_strategy="epoch",

            logging_strategy="epoch",

            learning_rate=LEARNING_RATE,

            weight_decay=WEIGHT_DECAY,

            per_device_train_batch_size=BATCH_SIZE,

            per_device_eval_batch_size=BATCH_SIZE,

            num_train_epochs=NUM_EPOCHS,

            warmup_ratio=WARMUP_RATIO,

            lr_scheduler_type="linear",

            optim="adamw_torch",

            seed=SEED,

            fp16=torch.cuda.is_available(),

            report_to="none",

            dataloader_num_workers=2,

            group_by_length=True,

            load_best_model_at_end=True,

            metric_for_best_model="accuracy",

            greater_is_better=True
        )

In [ ]:
        trainer = Trainer(

            model=model,

            args=training_args,

            train_dataset=encoded_dataset["train"],

            eval_dataset=encoded_dataset[validation_split],

            processing_class=tokenizer,

            data_collator=data_collator,

            compute_metrics=compute_metrics

        )

In [131]:
        print("\nTraining Started...\n")

        trainer.train()

        print("\nTraining Completed.\n")

In [132]:
        metrics = trainer.evaluate()

        experiment_time = (time.time() - experiment_start)/60

        print(f"Training Time : {experiment_time:.2f} minutes")

In [ ]:
############################################################
# GPU Memory Usage
############################################################

if torch.cuda.is_available():

    torch.cuda.synchronize()

    gpu_memory = torch.cuda.max_memory_allocated() / (1024**2)

else:

    gpu_memory = psutil.Process().memory_info().rss / (1024**2)

In [82]:
############################################################
# Inference Benchmark
############################################################

torch.cuda.empty_cache()

start = time.time()

prediction_output = trainer.predict(

    encoded_dataset[validation_split]

)

end = time.time()

total_samples = len(encoded_dataset[validation_split])

latency = ((end-start)/total_samples)*1000

throughput = total_samples/(end-start)



Total Experiments : 1
Total Time Taken  : 8.21 Minutes


In [1]:
############################################################
# Energy Consumption
############################################################

try:

    pynvml.nvmlInit()

    handle = pynvml.nvmlDeviceGetHandleByIndex(0)

    power = pynvml.nvmlDeviceGetPowerUsage(handle)/1000

    energy = power*(end-start)

    energy_per_sample = energy/total_samples

    pynvml.nvmlShutdown()

except:

    energy = None

    energy_per_sample = None

Hii


In [ ]:
############################################################
# Save Benchmark
############################################################

results.append({

    "Dataset":task.upper(),

    "Model":model_name,

    "Accuracy":metrics["eval_accuracy"],

    "Precision":metrics["eval_precision"],

    "Recall":metrics["eval_recall"],

    "F1-Score":metrics["eval_f1"],

    "Memory(MB)":gpu_memory,

    "Latency(ms)":latency,

    "Throughput":throughput,

    "Energy(J)":energy,

    "Energy/Sample(J)":energy_per_sample,

    "Training Time(min)":experiment_time

})

In [ ]:
############################################################
# Clear Memory
############################################################

del trainer

del model

del tokenizer

gc.collect()

torch.cuda.empty_cache()

In [ ]:
############################################################
# Final Results
############################################################

results_df = pd.DataFrame(results)

results_df = results_df.round(4)

display(results_df)

In [ ]:
############################################################
# Total Runtime
############################################################

total_time=(time.time()-overall_start)/60

print("="*80)

print(f"Total Runtime : {total_time:.2f} Minutes")

print("="*80)